### Mobile – Lösung Schritt für Schritt

*33. Bundeswettbewerb Informatik (2014), 1. Runde, Aufgabe 2*

Ein Mobile besteht aus waagerechten Balken, senkrechten Drähten und an Drähten hängenden Figuren. Die Gewichte der Balken und Drähte können vernachlässigt werden, während die Gewichte der Figuren die möglichen Formen des Mobiles beeinflussen.

Jeder Balken hat einen Aufhängepunkt und weitere Stellen, an denen tiefer hängende Balken oder Figuren an ihm befestigt sind. Der Balken ist balanciert, wenn die Gewichte der an ihm befestigten Strukturen, multipliziert mit den jeweiligen Abständen ihrer Befestigung vom Aufhängepunkt des Balkens, die Summe Null haben, falls Abstände auf einer Seite vom Aufhängepunkt mit 1, die auf der anderen Seite mit –1 multipliziert werden. Das gesamte Mobile ist balanciert, wenn alle seine Balken balanciert sind.

Zum Beispiel ist das abgebildete Mobile balanciert, denn: 1·2 + 2·(–1) = 0 und (1+2)·4 + 2·1 + 3·(–2) + 2·(–4) = 0.

#### Aufgabe 2

1. Erfinde und beschreibe eine einfache textuelle Darstellung von Mobiles, die eine Überprüfung der Balanciertheit erlaubt. Nicht für diesen Zweck relevante Aspekte, wie zum Beispiel die Länge der Drähte, sollen außer Acht gelassen werden. Zeige unter anderem das abgebildete Mobile in deiner Darstellung.
2. Schreibe ein Programm, das eine Folge w1, …, wn von Gewichten von n Figuren einliest und die textuelle Darstellung eines balancierten Mobiles mit den n Figuren ausgibt. Aus ästhetischen Gründen sollen an keinem Balken mehr als vier Strukturen befestigt sein.

*Freiwillige Zusatzaufgabe, ohne Einfluss auf die Bewertung:* Erweitere dein Programm so, dass es auch eine nette Zeichnung des gefundenen Mobiles produziert.

**Zum Bild:** Das abgebildete Mobile hat einen oberen Balken mit vier Strukturen: einen Unterbalken mit den Figuren 1 und 2 (Gesamtgewicht 3) im Abstand 4, Figur 2 im Abstand 1, Figur 3 im Abstand −2 und Figur 2 im Abstand −4. Der Unterbalken trägt Figur 1 im Abstand 2 und Figur 2 im Abstand −1.

**Zu den Daten:** Beispieldaten gibt es zu dieser Aufgabe nicht. In `eigene_testdaten/` liegen fünf Gewichtsfolgen (`gewichte1.txt` bis `gewichte5.txt`, Gewichte in einer Zeile) und drei Mobiles in der Darstellung dieses Notebooks (`mobile_aufgabenblatt.txt`, `mobile_hinweise1.txt`, `mobile_hinweise2.txt`).

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Bäume und Rekursion:** Ein Mobile ist ein Baum: Ein Balken trägt Figuren (Blätter) und andere Balken (Teilbäume). Das Gewicht eines Balkens kennt man erst, wenn man die Gewichte aller seiner Strukturen kennt, also muss man die tiefer hängenden Balken zuerst ausrechnen. Das macht eine **rekursive Funktion** von selbst.
- **Reihenfolge der Tiefensuche:** Schreibt man die Balken so auf, wie eine Tiefensuche sie besucht (erst der Balken, dann alles, was links unter ihm hängt, dann das nächste), braucht man keine Namen für die Balken.
- **Schwerpunkt:** Ein Balken ist im Gleichgewicht, wenn man ihn am Schwerpunkt aufhängt. Mit Gewichten `w_i` an den Stellen `x_i` liegt der Schwerpunkt bei `(Summe x_i · w_i) / (Summe w_i)`.
- **Zusammenfassen wie bei Huffman:** Aus vielen Teilen wird ein Baum, indem man immer wieder einige Teile zu einem neuen Teil zusammenfasst, der dann selbst wieder zur Auswahl steht.

**Nützliche Funktionen und Bibliotheken**
- `math.gcd(a, b)`: größter gemeinsamer Teiler, zum Kürzen der Abstände.
- `str.split()` und `' '.join(...)`: Textzeilen in Zahlen zerlegen und wieder zusammensetzen.

## Teil 1: Die textuelle Darstellung

Für einen Balken brauchen wir pro Struktur zwei Zahlen: ihr **Gewicht** `w` und ihren **Abstand** `a` vom Aufhängepunkt (negativ: links, positiv: rechts). Der Balken ist im Gleichgewicht, wenn die Summe aller `a · w` gleich 0 ist.

Bei einer Figur kennen wir das Gewicht. Bei einem Unterbalken ist es die Summe der Gewichte seiner Strukturen. Damit wir ihn nicht benennen müssen, schreiben wir **jeden Balken in eine eigene Zeile** und ordnen die Zeilen wie bei der Tiefensuche:

- Eine Zeile besteht aus Zahlenpaaren `w a`, von links nach rechts.
- Für eine **Figur** steht ihr Gewicht `w` (immer größer als 0).
- Für einen **Unterbalken** steht an der Stelle von `w` eine **0**. Welche Zeile gemeint ist, ergibt sich aus der Reihenfolge: Direkt nach der Zeile eines Balkens kommen die Zeilen des ersten Unterbalkens (mit allen seinen Unterbalken), danach die des zweiten und so weiter.

Das Mobile der Aufgabe sieht so aus (`eigene_testdaten/mobile_aufgabenblatt.txt`, von links nach rechts geordnet, also mit vertauschten Vorzeichen gegenüber der Rechnung in der Aufgabe; das Spiegelbild ist genauso balanciert):

In [ ]:
with open('eigene_testdaten/mobile_aufgabenblatt.txt') as datei:
    print(datei.read())

0 -4 2 -1 3 2 2 4
1 -2 2 1



Erste Zeile: Ein Unterbalken (`0`) im Abstand −4, eine Figur 2 im Abstand −1, eine Figur 3 im Abstand 2 und eine Figur 2 im Abstand 4. Zweite Zeile: der Unterbalken mit Figur 1 im Abstand −2 und Figur 2 im Abstand 1.

## Schritt 1: Balanciertheit prüfen

Wir lesen die Zeilen in eine Liste. Dann schreiben wir eine rekursive Funktion `pruefe_balken(zeilen, z)`: Sie prüft den Balken in Zeile `z` und liefert zwei Werte zurück, sein **Gewicht** (oder −1, wenn er oder ein Unterbalken nicht balanciert ist) und die Nummer der **nächsten Zeile**, die noch nicht verbraucht ist. Trifft sie auf ein `w = 0`, ruft sie sich für die nächste freie Zeile auf.

In [ ]:
def lies_zeilen(pfad):
    """Liest die Zeilen einer Mobile-Datei in eine Liste von Texten."""
    with open(pfad) as datei:
        return datei.read().splitlines()


def pruefe_balken(zeilen, z):
    """Prüft den Balken in Zeile z samt Unterbalken.
    Rückgabe: (Gewicht oder -1, nächste noch nicht gelesene Zeile)."""
    zahlen = [int(wort) for wort in zeilen[z].split()]
    naechste = z + 1
    gesamt = 0
    drehmoment = 0
    for i in range(0, len(zahlen), 2):
        w = zahlen[i]
        a = zahlen[i + 1]
        if w == 0:                                  # hier hängt ein Unterbalken
            w, naechste = pruefe_balken(zeilen, naechste)
            if w == -1:
                return -1, naechste
        gesamt += w
        drehmoment += a * w
    if drehmoment != 0:
        return -1, naechste
    return gesamt, naechste


def pruefe_mobile(zeilen):
    """Gesamtgewicht des Mobiles, oder -1 wenn es nicht balanciert ist."""
    gewicht, naechste = pruefe_balken(zeilen, 0)
    if naechste != len(zeilen):                     # Zeilen übrig: gehören zu keinem Balken
        return -1
    return gewicht


for name in ['mobile_aufgabenblatt', 'mobile_hinweise1', 'mobile_hinweise2']:
    zeilen = lies_zeilen('eigene_testdaten/' + name + '.txt')
    print(name, pruefe_mobile(zeilen))

mobile_aufgabenblatt 10
mobile_hinweise1 21
mobile_hinweise2 -1


Das Mobile der Aufgabe hat das Gesamtgewicht 10 (3 + 2 + 3 + 2). `mobile_hinweise1.txt` ist das Beispiel aus den Lösungshinweisen mit Gesamtgewicht 21. In `mobile_hinweise2.txt` ist dort eine Zahl geändert (`1 -3 3 1` statt `1 -4 4 1`), und die dritte Zeile ist nicht mehr balanciert: Wir bekommen −1.

## Teil 2: Ein Mobile bauen

### Schritt 2: Einen Balken im Gleichgewicht aufhängen

Gegeben sind die Gewichte `w_0 ... w_{k-1}` der Strukturen für einen Balken. Wir hängen sie zunächst im Abstand 1 nebeneinander auf, an den Stellen `x_i = i`. Der Schwerpunkt ist `X = S / W` mit `S = Summe x_i · w_i` und `W = Summe w_i`. Der Abstand vom Schwerpunkt ist `x_i - X`. Damit wir keine Brüche bekommen, multiplizieren wir alle Abstände mit `W`:

`a_i = W · x_i - S`

Dann ist `Summe a_i · w_i = W · S - S · W = 0`. Alle Abstände sind ganze Zahlen und paarweise verschieden. Damit sie nicht größer sind als nötig, teilen wir noch durch den größten gemeinsamen Teiler aller `a_i`.

In [ ]:
from math import gcd


def abstaende(gewichte):
    """Ganzzahlige Abstände für Strukturen mit diesen Gewichten (von links nach rechts), Balken im Gleichgewicht."""
    gesamt = sum(gewichte)
    moment = 0
    for i in range(len(gewichte)):
        moment += i * gewichte[i]
    a = []
    for i in range(len(gewichte)):
        a.append(gesamt * i - moment)
    teiler = 0
    for x in a:
        teiler = gcd(teiler, abs(x))
    return [x // teiler for x in a]


print(abstaende([1, 2]))
print(abstaende([2, 2, 3]))
print(abstaende([1, 2, 2, 2]))

[-2, 1]
[-8, -1, 6]
[-12, -5, 2, 9]


Probe: `[1, 2]` gibt `[-2, 1]` (1·(−2) + 2·1 = 0), bei `[2, 2, 3]` ist 2·(−8) + 2·(−1) + 3·6 = 0. Jede Gewichtsfolge geht, auch wenn alle Gewichte gleich sind (dann hängt der Balken in der Mitte).

### Schritt 3: Aus allen Figuren ein Mobile machen

Wir führen eine Liste `offen` mit den noch nicht aufgehängten Teilen. Ein Teil ist ein Paar aus **Gewicht** und **Zeilen**: Eine Figur hat die Zeilenliste `None` (kein Balken), ein Balken die Liste seiner Textzeilen (seine eigene Zeile, danach die seiner Unterbalken).

Solange mehr als ein Teil offen ist, nehmen wir die **leichtesten** Teile (höchstens vier) und hängen sie an einen neuen Balken. Das Ergebnis ist ein neues Teil, das wir wieder einsortieren. Die Liste halten wir nach dem Gewicht sortiert, deshalb stehen die leichtesten vorn. Damit hängen die leichten Figuren tief, und oben hängen die schweren Teile.

Damit möglichst jeder Balken vier Strukturen trägt, nimmt der erste Balken `2 + (n - 2) % 3` Teile (also 2, 3 oder 4), jeder weitere vier. Die Rechnung: Ein Balken mit `m` Teilen macht aus `m` Teilen eines, spart also `m - 1`. Von `n` Teilen auf 1 kommen heißt `n - 1` sparen. Mit vier Teilen je Balken spart man 3 pro Balken, ein Rest von 1 oder 2 bleibt für den ersten Balken (2 oder 3 Teile). Jeder Balken hat so mindestens zwei Strukturen. Gibt es nur **eine** Figur, ist das Mobile ein Balken mit dieser Figur im Abstand 0.

In [ ]:
def einsortieren(offen, teil):
    """Fügt das Paar (Gewicht, Zeilen) hinter allen leichteren oder gleich schweren Teilen ein."""
    stelle = 0
    while stelle < len(offen) and offen[stelle][0] <= teil[0]:
        stelle += 1
    offen.insert(stelle, teil)


def mache_balken(teile):
    """Hängt die Teile (Liste von Paaren (Gewicht, Zeilen)) an einen neuen Balken. Rückgabe: neues Teil."""
    gewichte = []
    for gewicht, zeilen in teile:
        gewichte.append(gewicht)
    a = abstaende(gewichte)
    eigene_zeile = []
    unterbalken = []
    for i in range(len(teile)):
        gewicht, zeilen = teile[i]
        if zeilen is None:                          # Figur
            eigene_zeile += [gewicht, a[i]]
        else:                                       # Unterbalken: Zeilen kommen direkt danach
            eigene_zeile += [0, a[i]]
            unterbalken += zeilen
    text = ' '.join(str(x) for x in eigene_zeile)
    return sum(gewichte), [text] + unterbalken


def konstruiere(gewichte):
    """Baut aus den Figurengewichten ein balanciertes Mobile, Rückgabe: Liste der Zeilen."""
    if len(gewichte) == 1:
        return [str(gewichte[0]) + ' 0']
    offen = []
    for w in gewichte:
        einsortieren(offen, (w, None))
    anzahl = 2 + (len(offen) - 2) % 3
    while len(offen) > 1:
        teile = offen[:anzahl]
        offen = offen[anzahl:]
        einsortieren(offen, mache_balken(teile))
        anzahl = 4
    return offen[0][1]


for w in [[7], [3, 4], [1, 2, 2, 2, 3]]:
    print(w)
    print('\n'.join(konstruiere(w)))
    print()

[7]
7 0

[3, 4]
3 -4 4 3

[1, 2, 2, 2, 3]
2 -17 2 -7 3 3 0 13
1 -2 2 1



Für `[1, 2, 2, 2, 3]` (Beispiel 1 der Hinweise) werden zuerst `1` und `2` zu einem Balken (Gewicht 3). Danach hängen `2`, `2`, `3` und dieser Balken am obersten Balken. Die Hinweise erzeugen die Mobiles zufällig, deshalb sehen ihre Ergebnisse anders aus. Wichtig ist nur, dass unser Mobile geprüft **balanciert** ist.

### Schritt 4: Alle Beispiele prüfen

Wir lesen die Gewichte aus den Dateien, bauen das Mobile und prüfen es mit `pruefe_mobile` aus Schritt 1. Zusätzlich zählen wir selbst nach, dass alle Figuren vorkommen und dass an keinem Balken mehr als vier Strukturen hängen. Dazu liest `kontrolle` die Textzeilen noch einmal und sammelt die Figurengewichte (`w > 0`).

In [ ]:
def lies_gewichte(pfad):
    with open(pfad) as datei:
        return [int(wort) for wort in datei.read().split()]


def kontrolle(zeilen):
    """Gibt (Figurengewichte sortiert, größte Zahl Strukturen an einem Balken, größter Abstand) zurück."""
    figuren = []
    am_meisten = 0
    weitester = 0
    for zeile in zeilen:
        zahlen = [int(wort) for wort in zeile.split()]
        am_meisten = max(am_meisten, len(zahlen) // 2)
        for i in range(0, len(zahlen), 2):
            if zahlen[i] > 0:
                figuren.append(zahlen[i])
            weitester = max(weitester, abs(zahlen[i + 1]))
    return sorted(figuren), am_meisten, weitester


for nr in range(1, 6):
    gewichte = lies_gewichte('eigene_testdaten/gewichte%d.txt' % nr)
    zeilen = konstruiere(gewichte)
    figuren, am_meisten, weitester = kontrolle(zeilen)
    ok = figuren == sorted(gewichte) and am_meisten <= 4
    print('gewichte%d: %d Figuren, %d Balken, Gesamtgewicht %d, höchstens %d Strukturen, größter Abstand %d, alles in Ordnung: %s' %
          (nr, len(gewichte), len(zeilen), pruefe_mobile(zeilen), am_meisten, weitester, ok))

gewichte1: 5 Figuren, 2 Balken, Gesamtgewicht 10, höchstens 4 Strukturen, größter Abstand 17, alles in Ordnung: True
gewichte2: 8 Figuren, 3 Balken, Gesamtgewicht 21, höchstens 4 Strukturen, größter Abstand 38, alles in Ordnung: True
gewichte3: 20 Figuren, 7 Balken, Gesamtgewicht 20, höchstens 4 Strukturen, größter Abstand 17, alles in Ordnung: True
gewichte4: 1 Figuren, 1 Balken, Gesamtgewicht 7, höchstens 1 Strukturen, größter Abstand 0, alles in Ordnung: True
gewichte5: 13 Figuren, 4 Balken, Gesamtgewicht 2579, höchstens 4 Strukturen, größter Abstand 4786, alles in Ordnung: True


Alle fünf Mobiles sind balanciert (das Gesamtgewicht ist die Summe der Figuren, nicht −1), enthalten genau die Figuren der Eingabe und haben höchstens vier Strukturen je Balken. Die Dateien sind:

- `gewichte1.txt`: `1 2 2 2 3` (Beispiel 1 der Hinweise)
- `gewichte2.txt`: `1 1 2 2 2 4 4 5` (Beispiel 2 der Hinweise)
- `gewichte3.txt`: zwanzigmal die 1
- `gewichte4.txt`: eine einzige Figur mit Gewicht 7
- `gewichte5.txt`: 13 Fibonacci-Zahlen von 3 bis 987 (große Unterschiede)

Das Mobile zu `gewichte2.txt` als Text:

In [ ]:
print('\n'.join(konstruiere(lies_gewichte('eigene_testdaten/gewichte2.txt'))))

4 -38 4 -17 5 4 0 25
2 -3 2 -1 2 1 0 3
1 -1 1 1


Die Abstände sind ganze Zahlen, aber teils recht groß (letzte Angabe in der Ausgabe oben). Bei sehr verschiedenen Gewichten (Fibonacci) wird das Mobile sehr breit. Für die Aufgabe ist das richtig, für eine „nette Zeichnung“ aber ungünstig. Im Abschnitt „Zum Schluss“ suchen wir mit MiniZinc bei festem Aufbau die kleinsten ganzzahligen Abstände.

## Zum Schluss: kleinere Abstände mit MiniZinc

Wir halten den **Aufbau** aus Python fest (welche Strukturen an welchem Balken hängen, und deren Gewichte) und lassen MiniZinc die Abstände wählen: ganzzahlig, an jedem Balken paarweise verschieden und im Gleichgewicht, mit möglichst kleinem **größten Abstand** `m`. Das Modell nutzt, was du aus den Arbeitsblättern kennst: ein 2D-Feld `w` (Balken mal Platz), `sum`, `forall` mit mehreren Laufvariablen, eine Entscheidungsvariable für das Maximum und `solve minimize`. Neu ist nur `->` (wenn ... dann): Ist ein Platz leer (`w = 0`), muss auch der Abstand 0 sein. Die Datendatei erzeugt eine Python-Zelle aus den Zeilen des Mobiles: `tabelle` liest die Zeilen wie `pruefe_balken` und merkt sich für jeden Balken die Gewichte seiner Strukturen. Als Obergrenze `R` nehmen wir den größten Abstand der Python-Lösung.

In [ ]:
def tabelle(zeilen, z, tab):
    """Wie pruefe_balken, sammelt aber in tab pro Balken (in Zeilenreihenfolge) die Gewichte seiner Strukturen."""
    zahlen = [int(wort) for wort in zeilen[z].split()]
    reihe = []
    tab.append(reihe)
    naechste = z + 1
    for i in range(0, len(zahlen), 2):
        w = zahlen[i]
        if w == 0:
            w, naechste = tabelle(zeilen, naechste, tab)
        reihe.append(w)
    return sum(reihe), naechste


def als_dzn(zeilen):
    """Datendatei für MiniZinc: Balken, Obergrenze R und Gewichtstabelle (freie Plätze mit 0 aufgefüllt)."""
    tab = []
    tabelle(zeilen, 0, tab)
    flach = []
    for reihe in tab:
        flach += reihe + [0] * (4 - len(reihe))
    weitester = kontrolle(zeilen)[2]
    return 'nb = %d;\nR = %d;\nw = array2d(1..nb, 1..4, %s);' % (len(tab), weitester, flach)


zeilen_eins = konstruiere(lies_gewichte('eigene_testdaten/gewichte1.txt'))
print(als_dzn(zeilen_eins))

nb = 2;
R = 17;
w = array2d(1..nb, 1..4, [2, 2, 3, 3, 1, 2, 0, 0]);


**Modell** (Datei `mobile.mzn`):

```minizinc
int: nb;                          % Anzahl Balken
int: R;                           % Obergrenze für die Abstände
array[1..nb, 1..4] of int: w;     % Gewicht der Struktur (0 = Platz frei)
array[1..nb, 1..4] of var -R..R: a;
var 0..R: m;                      % größter Abstand

constraint forall(b in 1..nb)(sum(s in 1..4)(a[b, s] * w[b, s]) = 0);
constraint forall(b in 1..nb, s in 1..4)(w[b, s] = 0 -> a[b, s] = 0);
constraint forall(b in 1..nb, s, t in 1..4 where s < t)(w[b, s] > 0 /\ w[b, t] > 0 -> a[b, s] != a[b, t]);
constraint forall(b in 1..nb, s in 1..4)(abs(a[b, s]) <= m);

solve minimize m;
output ["m = \(m)\n", "a = \([a[b, s] | b in 1..nb, s in 1..4])\n"];
```

**Daten** für `gewichte1.txt` (`mobile1.dzn`, in Python aus dem Mobile erzeugt):

```
nb = 2;
R = 17;
w = array2d(1..nb, 1..4, [2, 2, 3, 3, 1, 2, 0, 0]);
```

**Ausgabe** (Gecode, letzte und damit beste Lösung):

```
m = 2
a = [2, -2, 1, -1, 2, -1, 0, 0]
```

Pro Balken stehen vier Zahlen, ein freier Platz hat den Abstand 0. Der größte Abstand ist nur noch **2** statt 17. Probe in Python: Wir setzen die Abstände von MiniZinc in die Zeilen des Mobiles ein und prüfen es mit `pruefe_mobile`.

In [ ]:
mzn_abstaende = [2, -2, 1, -1, 2, -1, 0, 0]           # a aus der Ausgabe von MiniZinc
neu = []
for b in range(len(zeilen_eins)):
    zahlen = [int(wort) for wort in zeilen_eins[b].split()]
    for i in range(len(zahlen) // 2):
        zahlen[2 * i + 1] = mzn_abstaende[4 * b + i]
    neu.append(' '.join(str(x) for x in zahlen))
print('\n'.join(neu))
print(pruefe_mobile(neu))

2 2 2 -2 3 1 0 -1
1 2 2 -1
10


Gewicht 10, das Mobile ist balanciert und deutlich schmaler.

**Messungen bei allen fünf Beispielen** (Gecode, jeweils unter einer Sekunde; MiniZinc beweist, dass der Wert bei diesem Aufbau **optimal** ist):

| Beispiel | größter Abstand in Python | größter Abstand mit MiniZinc |
|---|---|---|
| gewichte1 | 17 | 2 |
| gewichte2 | 38 | 3 |
| gewichte3 | 17 | 3 |
| gewichte4 | 0 | 0 |
| gewichte5 | 4786 | 11 |

**Grenzen:** Mit Chuffed sind `gewichte1` bis `gewichte4` genauso schnell, bei `gewichte5` (Fibonacci-Gewichte, große Abstände) war der Solver nach 8 Sekunden nicht fertig und die beste Lösung bis dahin noch weit vom Optimum 11 entfernt; mit Gecode klappt auch das. MiniZinc ersetzt die Konstruktion in Python nicht: Es wählt nicht den Aufbau (wer mit wem an einem Balken hängt), sondern nur die Abstände bei festem Aufbau. Den Aufbau mit zu wählen wäre ein deutlich größeres Modell.

## Für Neugierige

- **Zufällige Mobiles:** Die Hinweise wählen die Strukturen und die Abstände zufällig (bis auf den letzten Abstand, der aus dem Gleichgewicht berechnet wird). So bekommt man bei jedem Aufruf ein anderes Mobile. Mit `random.sample` und `random.randint` ist das schnell gebaut.
- **Bruchzahlen statt ganzer Zahlen:** Wer die Abstände als Dezimalzahlen ausgibt, braucht den Trick mit der Multiplikation nicht, bekommt aber Rundungsfehler beim Prüfen. Mit ganzen Zahlen (oder `fractions.Fraction`) ist die Prüfung `== 0` exakt.
- **Zeichnung (Zusatzaufgabe):** Man kann das Mobile als Text zeichnen oder mit einer Grafikbibliothek. Wichtig ist, dass man die **Breite** eines Unterbalkens kennt, damit er nicht in einen Nachbarn ragt. Kleine ganzzahlige Abstände wie die von MiniZinc helfen dabei.
- **Balken mit nur einer Struktur** wären auch balanciert (Abstand 0), sie sind aber unschön. Unser Verfahren baut immer Balken mit mindestens zwei Strukturen, außer bei einer einzigen Figur.